In [1]:
import phonlp
from retrieval.src.extract_triplet import *


D:\uit_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_path = r"D:\uit_chatbot\retrieval\phonlp"
phoNLP_model = phonlp.load(save_dir=model_path)

Loading model from: D:\uit_chatbot\retrieval\phonlp/phonlp.pt


In [3]:
def init_vncorenlp(vncorenlp_dir, annotators=None):
    if annotators is None:
        annotators = ["wseg"] #["wseg", "pos", "ner", "parse"]
    if "rdrsegmenter" not in globals():
        import py_vncorenlp
        rdrsegmenter = py_vncorenlp.VnCoreNLP(
            annotators=annotators,
            save_dir=vncorenlp_dir
        )
    return rdrsegmenter

In [4]:
vncorenlp_client = init_vncorenlp(r"D:\uit_chatbot\graph\VnCoreNLP-1.2")

In [5]:
stopwords = load_stopwords(r"D:\uit_chatbot\graph\stopwords.csv")


In [6]:
import pandas as pd
from collections import defaultdict

# --- BƯỚC 1: ĐỊNH NGHĨA HÀM (Code của bạn) ---
def print_dependency_tree(df):
    """
    In ra cây phụ thuộc dạng văn bản từ một DataFrame.
    """
    child_map = defaultdict(list)

    # Lỗi tiềm ẩn: Đảm bảo 'head' và 'id' là kiểu int
    # Nếu df được đọc từ file, 'head' có thể là float (ví dụ: 4.0)
    # Chúng ta ép kiểu để đảm bảo an toàn
    try:
        df['id'] = df['id'].astype(int)
        df['head'] = df['head'].astype(int)
    except Exception as e:
        print(f"Cảnh báo: Không thể ép kiểu cột 'id'/'head': {e}")

    for _, row in df.iterrows():
        # Chuyển hàng sang dict để dễ truy cập
        child_map[row['head']].append(row.to_dict())

    # --- Định nghĩa hàm đệ quy để in cây ---
    def print_tree(head_id, prefix):
        children = child_map.get(head_id, [])
        num_children = len(children)

        for i, child_info in enumerate(children):
            is_last_child = (i == num_children - 1)

            connector = "└── " if is_last_child else "├── "

            deprel = child_info['deprel']
            word = child_info['word']
            child_id = int(child_info['id']) # Đã ép kiểu ở trên nhưng an toàn
            pos = child_info['pos']

            print(f"{prefix}{connector}[{deprel}] {word} (id={child_id}, pos={pos})")

            new_prefix = prefix + ("    " if is_last_child else "│   ")

            # Gọi đệ quy cho "con" này (giờ nó là "mẹ")
            print_tree(child_id, new_prefix)

    # --- Bắt đầu in từ gốc (ROOT) ---
    print("[ROOT] (id=0)")
    # Gọi hàm bắt đầu từ head_id=0 (tương ứng với 'ROOT')
    print_tree(head_id=0, prefix="")


In [7]:
#from graph.src.triplet_extraction import *
def extract_triplets(summarized_text):
    sentences = summarized_text.split('.')
    sentences = [cau.strip() for cau in sentences if cau.strip()]
    triplets_list = []
    for sentence in sentences:
        sentence = clean_text(sentence)
        segmented_text = vncorenlp_client.word_segment(sentence)
        # Stopword filtering
        parts = segmented_text[0].split(" ")
        filtered_parts = [part for part in parts if is_valid_term(part, stopwords)]
        filtered_text = " ".join(filtered_parts)
        # Annotate the filtered text
        annotation = phoNLP_model.annotate(text=filtered_text)
        df = parsing_result(annotation)
        print("---df---")
        print(df)
        print("---dependency tree---")
        print_dependency_tree(df)
        print("------")
        result = process_sentence(df)
        if result is None:
            result = []

        triplets_list += [
            {
                "c1": c1,
                "r": r,
                "c2": c2
            }
            for (c1, r, c2) in result
            if c1 and r and c2
        ]
    return triplets_list

In [8]:
extract_triplets('Người lái xe máy vượt đèn đỏ.')

100%|██████████| 1/1 [00:00<00:00,  4.78it/s]

---df---
   id    word pos head deprel
0   1   người   N    4    sub
1   2  lái_xe   V    1   nmod
2   3     máy   N    1   nmod
3   4    vượt   V    0   root
4   5  đèn_đỏ   N    4    dob
---dependency tree---
[ROOT] (id=0)
└── [root] vượt (id=4, pos=V)
    ├── [sub] người (id=1, pos=N)
    │   ├── [nmod] lái_xe (id=2, pos=V)
    │   └── [nmod] máy (id=3, pos=N)
    └── [dob] đèn_đỏ (id=5, pos=N)
------


[{'c1': 'người', 'r': 'lái_xe', 'c2': 'máy'},
 {'c1': 'máy', 'r': 'vượt', 'c2': 'đèn_đỏ'}]